# 04 Data Integration

For this notebook, it will combining the cleaned datasets from Yahoo Finance, Wikipedia, and FRED into a shared project dataset. The goal is to prepare on clean set of files that can then be used for feature engiener, modeling, portfolio optimization, and dashboard work.

# Imports

In [2]:
import pandas as pd
import numpy as np 
from pathlib import Path

# Paths

In [3]:
YFINANCE_PATH = Path('../data/processed/yfinance')
WIKIPEDIA_PATH = Path('../data/processed/wikipedia')
FRED_PATH = Path('../data/processed/fred')
INTEGRATED_PATH = Path('../data/processed/integrated')

In [4]:
if not INTEGRATED_PATH.exists():
    INTEGRATED_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"{INTEGRATED_PATH} already exists.")

..\data\processed\integrated already exists.


# Load Cleaned Data

In [5]:
adj_close = pd.read_csv(
    YFINANCE_PATH / 'yfinance_adjusted_close_clean.csv',
    index_col=0,
    parse_dates=True
)

In [6]:
daily_returns = pd.read_csv(
    YFINANCE_PATH / 'yfinance_daily_returns_clean.csv',
    index_col=0,
    parse_dates=True
)

In [12]:
summary_stats = pd.read_csv(
    YFINANCE_PATH / "yfinance_processed_summary_stats.csv",
    index_col=0
)

In [13]:
risk_free_rate = pd.read_csv(
    FRED_PATH / "fred_risk_free_rate_processed.csv",
    index_col="date",
    parse_dates=True
)

In [14]:
cpi = pd.read_csv(
    FRED_PATH / "fred_cpi_processed.csv",
    index_col="date",
    parse_dates=True
)

In [15]:
sp500_metadata = pd.read_csv(
    WIKIPEDIA_PATH / "wikipedia_sp500_constituents_clean.csv"
)

# Inspect Data Shapes

In [16]:
print("Adjusted close:", adj_close.shape)
print("Daily returns:", daily_returns.shape)
print("Summary stats:", summary_stats.shape)
print("Risk-free rate:", risk_free_rate.shape)
print("CPI:", cpi.shape)
print("S&P 500 metadata:", sp500_metadata.shape)

Adjusted close: (2010, 21)
Daily returns: (2009, 21)
Summary stats: (21, 6)
Risk-free rate: (2010, 2)
CPI: (2892, 2)
S&P 500 metadata: (503, 3)


Now all cleaned datasets are loaded from their folders. The next step is to check whether the dates and ticker symbols are lined up correctly

# Check Date Alignment

In [17]:
print("Adjusted close date range:", adj_close.index.min(), "to", adj_close.index.max())
print("Daily returns date range:", daily_returns.index.min(), "to", daily_returns.index.max())
print("Risk-free rate date range:", risk_free_rate.index.min(), "to", risk_free_rate.index.max())
print("CPI date range:", cpi.index.min(), "to", cpi.index.max())

Adjusted close date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
Daily returns date range: 2018-01-03 00:00:00 to 2025-12-30 00:00:00
Risk-free rate date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
CPI date range: 2018-01-01 00:00:00 to 2025-12-01 00:00:00


In [21]:
print("Prices and risk-free rate same dates:", adj_close.index.equals(risk_free_rate.index))

# NOTE: Daily returns only need to be in the risk-free rate dates, not necessarily the same dates as prices
print("Daily returns dates are in risk-free rate:", daily_returns.index.isin(risk_free_rate.index).all())

Prices and risk-free rate same dates: True
Daily returns dates are in risk-free rate: True


The risk-free rate data is asligned to the same trading dates as the yfinance price data. Dialy returns start one day later because returns require a previous price to calculate the return. Therefore, the daily returns data will not have the same dates as the price data, but it should be a subset of the risk-free rate dates.

# Create Asset Metadata